<a href="https://colab.research.google.com/github/BarGinger/HCML-NLP-Project/blob/main/src/proto-lm/drugs_reviews_proto_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Using Proto-ML [1] to analyse the drug reviews dataset [2]

[1] https://github.com/yx131/proto-lm/tree/main

[2] https://www.kaggle.com/datasets/mohamedabdelwahabali/drugreview/data

### Global imports

In [ ]:
import argparse

import torch
from torch.utils.data import DataLoader
!pip install pytorch-lightning
import pytorch_lightning as pl
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
from ProtoLM import proto_lm
from proto_data_class import sst_datamodule
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers.tensorboard import TensorBoardLogger
!pip install -U datasets
import datasets

In [ ]:
from transformers import AutoConfig
import os

# Parameters for Proto-LM training on the drug review dataset
model_name = 'bert-base-uncased'  # Backbone LLM model to load
# Maximum sentence length to pad/truncate to
args = {
    'model_name': model_name,        # backbone LLM model to load
    'max_seq_length': 100,                # maximum sentence length to pad/truncate to
    'num_prototypes': 1000,               # number of prototypes to train
    'hidden_shape': 1024,                # hidden shape of each prototype, should match LLM output
    'num_classes': 1,                    # Changed to 1 for regression (predicting continuous rating 1-10)
    'cohsep_ratio': 0.5,                 # ratio of prototypes in class to push/pull
    'lambda0': 0.5,                      # lambda0 in loss
    'lr': 3e-4,                          # initial learning rate
    'proto_training_weights': 1,         # whether to train prototype weights (1=True, 0=False)
    'batch_size': 128,                   # batch size for dataloader
    'logger_dir': 'tb_logs',             # directory for the logger to store training details
    'checkpoint_dir': 'ckpt_dir',        # directory to store checkpoints
    'config_subdir': 'config_subdir',    # subdirectory for checkpoints of a certain config
    'max_epochs': 2,                     # number of epochs to train
    'num_gpu': 3,                        # number of gpus to train on
    'load_model': model_name,  # path to load a pretrained model, if any
}

# Dynamically fetch hidden size from the model configuration
config = AutoConfig.from_pretrained(args['model_name'])
hidden_size = config.hidden_size  # Dynamically get the hidden size (768 for bert-base-uncased)
args['hidden_shape'] = hidden_size

print(f'args: {args}')

# get data module
drug_review_dm = sst_datamodule(
    model_name_or_path=args['model_name'],
    max_seq_length=args['max_seq_length'],
    train_batch_size=args['batch_size'],
    eval_batch_size=args['batch_size']
)

# Check if data files exist and are not empty before setup
data_files = {
    "train": "/content/Data/drug_review_train_with_sentiment.csv",
    "validation": "/content/Data/drug_review_validation_with_sentiment.csv",
    "test": "/content/Data/drug_review_test_with_sentiment.csv"
}

missing_or_empty_files = []
for split, file_path in data_files.items():
    if not os.path.exists(file_path):
        missing_or_empty_files.append(f"{file_path} (Missing)")
    elif os.path.getsize(file_path) == 0:
        missing_or_empty_files.append(f"{file_path} (Empty)")
    # else:
    #     # Check the first few lines
    #     try:
    #         with open(file_path, 'r') as f:
    #             print(f"First 5 lines of {file_path}:")
    #             for i in range(5):
    #                 print(f.readline().strip())
    #     except Exception as e:
    #         print(f"Could not read {file_path}: {e}")


if missing_or_empty_files:
    print("Error: The following data files are missing or empty:")
    for item in missing_or_empty_files:
        print(item)
else:
    drug_review_dm.setup(stage='fit')

print(f"loading a model: {args['load_model']}")

base_model = AutoModelForSequenceClassification.from_pretrained(
    args['model_name'], ignore_mismatched_sizes=True
)

if hasattr(base_model, "roberta"):
    llm_model = base_model.roberta
elif hasattr(base_model, "bert"):
    llm_model = base_model.bert
else:
    llm_model = base_model

proto = proto_lm(
    pretrained_model=llm_model,
    max_seq_length=args['max_seq_length'],
    num_prototypes=args['num_prototypes'],
    hidden_shape=args['hidden_shape'],
    num_classes=args['num_classes'],
    cohsep_ratio=args['cohsep_ratio'],
    lambda0=args['lambda0'],
    lr=args['lr'],
    proto_training_weights=bool(args['proto_training_weights']),
)

In [ ]:
import os

# Assuming the 'Data' directory is directly under /content in Colab
base_dir = "/content/Data"
print(base_dir)

In [ ]:
# get training utilities like logger and checkpoints
from pytorch_lightning.loggers.tensorboard import TensorBoardLogger
from pytorch_lightning.callbacks import ModelCheckpoint

tb_logger = TensorBoardLogger(f"{args['logger_dir']}", name="drug_review_tensorboard_logs")
ckpt_path = f"{args['checkpoint_dir']}/{args['config_subdir']}"
checkpoint_callback = ModelCheckpoint(
    dirpath=ckpt_path,
    monitor='val_loss',
    save_top_k=3,
    filename="{epoch}-{val_loss:.4f}-{val_accuracy:.4f}"
)

# get trainer object
trainer = pl.Trainer(
    max_epochs=args['max_epochs'],
    accelerator="auto",
    devices=1,  # or just remove this line for auto
    logger=tb_logger,
    callbacks=[checkpoint_callback]
)

trainer.fit(proto, datamodule=drug_review_dm)

# Optionally test or save misclassified
# trainer.test(proto, datamodule=drug_review_dm, ckpt_path=args['load_model'])
# torch.save(proto.misclassified, 'drug_review_logs/misclassed.pt')

### Save the trained model

In [ ]:
# Save the final model weights
torch.save(proto.state_dict(), "final_proto_model.pt")

### Calc Quantus Metrics

In [ ]:
import quantus
import torch
import numpy as np
import pandas as pd

# Define your model and data
model = proto  # Your Proto-LM model
model_name_for_csv = f"ProtoLM_{model_name}"  # Add your model name here
model.eval()  # Set the model to evaluation mode

# Define a wrapper for your model to work with Quantus
class ModelWrapper:
    def __init__(self, model):
        self.model = model

    def __call__(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.model.LLM(
                input_ids=input_ids,
                attention_mask=attention_mask,
                output_hidden_states=True
            )
            logits = outputs.logits  # Assuming logits are the output
        return logits

wrapped_model = ModelWrapper(model)

# Define a sample batch of data (input_ids and attention_mask)
batch = next(iter(drug_review_dm.test_dataloader()))
input_ids = batch["input_ids"]
attention_mask = batch["attention_mask"]
sentiment_features = torch.cat((
    batch["sentiment_neg"],
    batch["sentiment_neu"],
    batch["sentiment_pos"],
    batch["sentiment_compound"]
), dim=1)
labels = batch["labels"]

# Pass sentiment features to the model
outputs = proto(
    input_ids=input_ids,
    attention_mask=attention_mask,
    sentiment_features=sentiment_features,
    labels=labels
)

# Define an attribution method (e.g., Integrated Gradients)
from captum.attr import IntegratedGradients
ig = IntegratedGradients(wrapped_model)

# Generate attributions for the input
attributions = ig.attribute(inputs=input_ids, additional_forward_args=(attention_mask,), target=labels)

# Convert attributions to numpy for Quantus
attributions_np = attributions.detach().cpu().numpy()

# Define Quantus metrics
metrics = {
    "Sparsity": quantus.Sparsity(),
    "Complexity": quantus.Complexity(),
    "Faithfulness": quantus.FaithfulnessCorrelation(),
    "Robustness": quantus.LocalLipschitzEstimate(),
    "Sensitivity": quantus.SensitivityN()
}

# Evaluate metrics
results = {}
for metric_name, metric in metrics.items():
    result = metric(
        model=wrapped_model,
        x_batch=input_ids.cpu().numpy(),
        y_batch=labels.cpu().numpy(),
        a_batch=attributions_np,
        explain_func=lambda x: attributions_np  # Use precomputed attributions
    )
    results[metric_name] = result

# Print results
for metric_name, result in results.items():
    print(f"{metric_name}: {result}")

# Export results to CSV
results_df = pd.DataFrame.from_dict(results, orient="index", columns=["Score"])
results_df["Model"] = model_name_for_csv  # Add model name to the DataFrame
results_df.reset_index(inplace=True)
results_df.rename(columns={"index": "Metric"}, inplace=True)

# Save to CSV
csv_filename = f"quantus_metrics_{model_name_for_csv}.csv"
results_df.to_csv(csv_filename, index=False)
print(f"Results saved to {csv_filename}")

### Plot similarity between test set cases and the learned concepts similar to figure 3 in their paper

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F

# Get prototype vectors (concepts)
prototypes = proto.prototypes.detach().cpu().numpy()  # shape: (num_prototypes, hidden_dim)

# Get representations for some samples (e.g., from your test set)
batch = next(iter(drug_review_dm.test_dataloader()))
with torch.no_grad():
    # Get last hidden states for the batch
    llm_out = proto.LLM(
        input_ids=batch['input_ids'].to(proto.device),
        attention_mask=batch['attention_mask'].to(proto.device),
        output_hidden_states=True
    )
    sample_reps = llm_out.hidden_states[-1][:, 0, :].cpu().numpy()  # [CLS] token or mean pooling



# prototypes: (num_prototypes, hidden_dim)
# sample_reps: (batch_size, hidden_dim)
# Assume proto.prototype_class_vec exists and is (num_prototypes, num_classes)

# 1. Identify positive and negative prototypes
# If proto.prototype_class_vec is one-hot or softmax over classes:
proto_class = proto.prototype_class_vec.detach().cpu().numpy()  # (num_prototypes, num_classes)
positive_proto_idx = np.argmax(proto_class[:, 1])  # class 1 = positive
negative_proto_idx = np.argmax(proto_class[:, 0])  # class 0 = negative

positive_proto = torch.tensor(prototypes[positive_proto_idx])
negative_proto = torch.tensor(prototypes[negative_proto_idx])

# 2. Compute similarities for each sample
sample_vecs = torch.tensor(sample_reps)  # (batch_size, hidden_dim)
sim_pos = F.cosine_similarity(sample_vecs, positive_proto.unsqueeze(0), dim=1)
sim_neg = F.cosine_similarity(sample_vecs, negative_proto.unsqueeze(0), dim=1)

# 3. Get ground-truth labels for the batch
labels = batch['labels'].cpu().numpy()

# 4. Plot
plt.figure(figsize=(8, 8))
for label in np.unique(labels):
    idxs = np.where(labels == label)[0]
    plt.scatter(sim_pos[idxs], sim_neg[idxs], label=f"Class {label}", alpha=0.7)
plt.xlabel("Similarity to Positive Prototype")
plt.ylabel("Similarity to Negative Prototype")
plt.title("2D Prototypical Space (like Proto-LM Fig. 3)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import torch.nn.functional as F

sample_vec = sample_reps[0]  # pick one sample
proto_vecs = torch.tensor(prototypes)
similarities = F.cosine_similarity(torch.tensor(sample_vec).unsqueeze(0), proto_vecs)
plt.bar(range(len(similarities)), similarities.numpy())
plt.xlabel("Prototype Index")
plt.ylabel("Cosine Similarity")
plt.title("Sample-Prototype Similarity")
plt.show()

In [ ]:
import torch
import pandas as pd
import numpy as np
import torch.nn.functional as F

# 1. Get all review texts and their embeddings from the training set
train_dataset = drug_review_dm.dataset["train"]
all_texts = train_dataset["review"]

# Get all input_ids and attention_mask for the train set
input_ids = train_dataset["input_ids"]
attention_mask = train_dataset["attention_mask"]

# Compute all embeddings (CLS token)
all_embeddings = []
batch_size = 128
for i in range(0, len(input_ids), batch_size):
    batch_input_ids = input_ids[i:i+batch_size].to(proto.device)
    batch_attention_mask = attention_mask[i:i+batch_size].to(proto.device)
    with torch.no_grad():
        outputs = proto.LLM(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            output_hidden_states=True
        )
        batch_embeds = outputs.hidden_states[-1][:, 0, :].cpu()  # CLS token
        all_embeddings.append(batch_embeds)
all_embeddings = torch.cat(all_embeddings, dim=0)  # (num_samples, hidden_dim)

# 2. For each prototype, find the closest text
prototypes = proto.prototypes.detach().cpu()  # (num_prototypes, hidden_dim)
closest_texts = []
for proto_vec in prototypes:
    sims = F.cosine_similarity(all_embeddings, proto_vec.unsqueeze(0), dim=1)
    idx = torch.argmax(sims).item()
    closest_texts.append(all_texts[idx])

# 3. Save to CSV
df = pd.DataFrame({"prototype_index": range(len(closest_texts)), "closest_text": closest_texts})
df.to_csv("prototype_texts.csv", index=False)